# 08 Aquifer Geology

**Series:** Tribal Soils and Geology

## Aquifer Geology of Pine Ridge

Groundwater on Pine Ridge is not abundant, and what is available
is controlled almost entirely by the stratigraphic sequence described in
notebooks 03 and 04. Three aquifer systems are relevant to the study area:

**Ogallala Group (Arikaree Formation) Primary aquifer:**
Miocene age sand and gravel that caps remnant buttes and mesas in the
western study area and thins eastward. This is the primary water supply
for much of western Pine Ridge. The 3D model provides, for the first time,
a continuous picture of the Ogallala top elevation across the reservation 
enabling systematic assessment of where the aquifer is accessible.

**Niobrara Formation Secondary aquifer:**
Fractured chalk and chalky shale. Provides water in some areas where the
Ogallala is absent, but yields are typically low and water quality is
variable.

**Madison Group Deep confined aquifer:**
Mississippian carbonate rocks at depths of 300-900m across the study area.
Artesian conditions in some locations. High dissolved solids limit use for
potable supply without treatment, but the Madison is an important stock
water source in some areas.

**The Pierre Shale as a confining layer:**
The Pierre Shale separates the near-surface system from deeper aquifers.
Its very low permeability makes it an effective aquitard protecting
deeper aquifers from surface contamination but also limiting recharge.

This notebook is the primary connection between the soils/geology series
and the tribal_water_monitoring series.

In [1]:
# Imports
import sys
from pathlib import Path
REPO_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent/"src").is_dir() and (parent/"config"/"config.yaml").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from the tribal-soils-geology repository or a subdirectory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import warnings, numpy as np, pandas as pd
import geopandas as gpd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, contextily as ctx, yaml
from shapely.geometry import box as sbox
from src.constants import (
    CRS_GEOGRAPHIC, CRS_PROJECTED, CRS_WEB, REPO_ROOT as _REPO_ROOT,
    OUTPUTS_DIR, FIGURES_DIR, PINE_RIDGE_BBOX,     STUDY_BBOX, WSD_3D_MODEL, WSD_KEY_UNITS,
)
from src.loaders import load_tribal_boundaries, load_usgs_well_sites
from src.sovereignty import print_data_acknowledgment, generate_citations
warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline
with open(_REPO_ROOT/"config"/"config.yaml") as f: CONFIG = yaml.safe_load(f)
TEAL="#007A6E"; TEAL_LT="#E0F4F2"; GRAY="#566573"; TERRACOTTA="#C0392B"
def despine(ax):
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
primary = load_tribal_boundaries(["Pine Ridge"])
print(f"Ready. Nations: {len(primary)}")

Ready. Nations: 1


Model-layer inventory is maintained in Notebook 04. This notebook opens only the three aquifer-relevant rasters to avoid repeatedly mixing FileGDB vector and raster drivers in one Windows kernel.

In [2]:
# Print data acknowledgement at the top of every notebook
print_data_acknowledgment(source_keys=["usgs_nwis_wells","usgs_3d_model"])

TRIBAL SOILS AND GEOLOGY DATA GOVERNANCE ACKNOWLEDGMENT

This analysis uses data that describes the lands and subsurface
resources of the Oglala Sioux Tribe and the Oglala Lakota people.
peoples. This data is governed by the following frameworks:

OCAP®  : Tribal Nations have the right to Ownership, Control,
         Access, and Possession of data about their lands,
         including subsurface geological and soil data.
         Reference: https://fnigc.ca/ocap-training/

CARE   : Data use must deliver Collective Benefit to Indigenous
         peoples, respect their Authority to Control, uphold
         Responsibility to communities, and center Ethics.
         Reference: https://www.gida-global.org/care

FAIR   : Data is Findable, Accessible, Interoperable, Reusable.
         FAIR governs technical standards; CARE and OCAP® govern
         the ethical obligations FAIR alone does not address.
         Reference: https://www.go-fair.org/fair-principles/

IEEE 2890-2025 : Recommended Pr

## USGS Well Coverage and Monitoring Gap Analysis

In [3]:
# Live NWIS access is optional so the geology workflow is reproducible offline.
# Set this to True only when you intentionally want to refresh the public-well
# inventory; a failed or empty query is not evidence about the cause of a gap.
RUN_LIVE_WELL_QUERY = False

if RUN_LIVE_WELL_QUERY:
    from src.loaders import load_usgs_well_sites
    print("Refreshing the public USGS groundwater-site inventory...")
    wells_pr = load_usgs_well_sites(PINE_RIDGE_BBOX)
    print(f"Pine Ridge bounding-box sites: {len(wells_pr)}")
    print("These are bounding-box inventory counts, not a causal monitoring-equity analysis.")
else:
    wells_pr = gpd.GeoDataFrame()
    print("Live NWIS query skipped (RUN_LIVE_WELL_QUERY=False).")
    print("No claim about current well coverage is made in this run.")


Live NWIS query skipped (RUN_LIVE_WELL_QUERY=False).
No claim about current well coverage is made in this run.


In [4]:
# Map only a deliberately refreshed inventory; never imply that an unrun query
# or a network failure represents zero monitoring sites.
if not RUN_LIVE_WELL_QUERY:
    print("Well map skipped because the live public inventory was not refreshed.")
elif wells_pr.empty:
    print("No mappable records were returned; no coverage inference is made.")
else:
    fig, ax = plt.subplots(figsize=(12, 9))
    primary.to_crs(CRS_WEB).plot(
        ax=ax, facecolor=TEAL_LT, edgecolor=TEAL,
        linewidth=2, alpha=0.3, zorder=2,
    )
    for wells, label, color in [(wells_pr, "Pine Ridge query", "#2166AC")]:
        if not wells.empty:
            wells.to_crs(CRS_WEB).plot(ax=ax, color=color, markersize=18, label=label, zorder=3)
    ax.set_title("Public USGS groundwater sites returned by this deliberate refresh")
    ax.legend()
    ax.set_axis_off()
    plt.show()


Well map skipped because the live public inventory was not refreshed.


## Aquifer Geometry from the 3D Model

In [8]:
from src.loaders import load_wsd_horizon_raster, resolve_wsd_horizon_layer

# The regional model aggregates the Arikaree Group within "pre-Ogallala";
# it does not provide a formation-specific Arikaree aquifer top.
AQUIFER_UNITS = {
    "pre-Ogallala": (
        "Aggregated unit includes Arikaree Group; use as regional context only, "
        "not as an Arikaree-specific aquifer surface"
    ),
    "Niobrara Formation": "Secondary aquifer in locally fractured chalk",
    "Madison Group": "Deep confined carbonate aquifer system",
}

print("AQUIFER UNIT STATUS USGS 3D MODEL")
for unit, description in AQUIFER_UNITS.items():
    layer = resolve_wsd_horizon_layer(unit)
    ds = load_wsd_horizon_raster(unit)
    print()
    print(f"{unit} -> {layer}")
    print(f"  {description}")
    if ds is None:
        print("  Raster unavailable; see the preceding warning for the specific cause.")
        continue
    with ds:
        data = ds.read(1, masked=True)
        print(f"  Modeled top elevation: {data.min():.1f} to {data.max():.1f} m NAVD88")
        print(f"  Grid size: {data.shape[0]} x {data.shape[1]}")


AQUIFER UNIT STATUS USGS 3D MODEL

pre-Ogallala -> WSD_TopPreOgallala
  Aggregated unit includes Arikaree Group; use as regional context only, not as an Arikaree-specific aquifer surface
  Modeled top elevation: 540.2 to 1118.0 m NAVD88
  Grid size: 1402 x 1565

Niobrara Formation -> WSD_TopNiobraraFormation
  Secondary aquifer in locally fractured chalk
  Modeled top elevation: -129.1 to 1275.7 m NAVD88
  Grid size: 1409 x 1798

Madison Group -> WSD_TopMadisonGroup
  Deep confined carbonate aquifer system
  Modeled top elevation: -1251.8 to 2172.7 m NAVD88
  Grid size: 1409 x 1798


## Tribal Well Log Data

In [6]:
from src.loaders import load_tribal_well_logs
from src.soil_evidence import validate_governed_records, GovernanceError

print("Loading governed well logs...")
candidate_wells = load_tribal_well_logs()
try:
    tribal_wells = validate_governed_records(candidate_wells, "aquifer-geology")
except GovernanceError as exc:
    tribal_wells = pd.DataFrame()
    print(f"Governance gate stopped well-log analysis: {exc}")

if tribal_wells.empty:
    print("No well-log records are authorized for aquifer-geology. No summary or map is produced.")
else:
    print(f"Authorized well-log records: {len(tribal_wells):,}")
    print("These observations describe sampled locations only.")


Loading governed well logs...
No well-log records are authorized for aquifer-geology. No summary or map is produced.


C:\Users\gekek\AppData\Local\Temp\ipykernel_36656\4229176771.py:5: UserWarning: No Tribal well log data found in data/governed/. See Field data forms/well_log_template.xlsx.
  candidate_wells = load_tribal_well_logs()


## Connection to tribal_water_monitoring

The aquifer geology documented here is the subsurface explanation for the
surface water and groundwater patterns analyzed in the tribal_water_monitoring
series. Key linkages:

**Ogallala thickness controls baseflow:** Where the Ogallala is thick and
saturated, it sustains dry-season baseflow in streams. Where it has been
depleted or is absent, streams are intermittent. The depth-to-Ogallala
map from the 3D model directly explains the streamflow reliability patterns
in tribal_water_monitoring notebook 03.

**Pierre Shale controls recharge:** The very low permeability of the Pierre
Shale limits recharge to the Madison and deeper aquifers. Areas where the
Ogallala is thin or absent and the Pierre Shale is near-surface have
essentially no groundwater recharge a permanent water scarcity condition
that land management cannot change.

**Fault compartmentalization:** The 35 faults in the 3D model may create
hydraulic barriers that compartmentalize the Ogallala aquifer. Areas
on different sides of a fault may not be hydraulically connected, meaning
pumping in one area does not affect the other and vice versa, recovery
in one compartment does not benefit adjacent areas.

In [7]:
# Print citations
print(generate_citations(["usgs_nwis_wells","usgs_3d_model","census_aiannh"]))

DATA CITATIONS

USGS NWIS Well Logs and Groundwater Data
  U.S. Geological Survey, 2024, National Water Information System data available on the World Wide Web (USGS Water Data for the Nation). https://waterdata.usgs.gov/nwis/
  https://waterdata.usgs.gov/nwis/
  Steward: US Geological Survey | License: Public domain

USGS 3D Geological Model of Western South Dakota
  Spangler, L.R., 2024, Digital data for a 3D Geological Model of western South Dakota, USA: U.S. Geological Survey data release, https://doi.org/10.5066/P9LK4QHJ.
  https://doi.org/10.5066/P9LK4QHJ
  Steward: US Geological Survey, Rocky Mountain Region | License: CC0 1.0 Universal (Public Domain)

US Census Bureau TIGER/Line AIANNH Boundaries
  US Census Bureau. TIGER/Line Shapefiles: American Indian / Alaska Native / Native Hawaiian Areas (AIANNH). https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html
  https://www.census.gov/geographies/mapping-files/
  Steward: US Census Bureau | License: